# Tools

In [ ]:
import os
from langchain.chat_models import init_chat_model 


os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

gemini_model = init_chat_model("google_genai:gemini-3.5-flash-lite")
response = gemini_model.invoke("did nasa create artificial rain cloud")
response 

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content=[{'type': 'text', 'text': 'The short answer is **no, NASA did not create artificial rain clouds.** \n\nHowever, this is a common misconception, and it usually stems from a few different things NASA *has* done related to rockets, weather, and clouds. Here is the truth behind the rumors:\n\n### 1. Rocket Launches Create "Artificial" Clouds (But Not Rain)\nWhen NASA launches large rockets (like the Space Launch System or the old Space Shuttle), the enormous amounts of water vapor and exhaust released into the atmosphere often condense to form large, billowy white clouds. \n* Sometimes, if the upper atmosphere is very cold and humid, these rocket launches can trigger the formation of **noctilucent (night-shining) clouds** or ice crystals. \n* However, these are byproducts of the launch, not a deliberate attempt to make rain. In fact, NASA tries to avoid bad weather during launches.\n\n### 2. NASA Studies Clouds and Weather Modification\nNASA has dozens of satellites in or

In [13]:
from langchain.tools import tool 

@tool
def get_weather(location:str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools = gemini_model.bind_tools([get_weather])

In [14]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'call_7502490': 'El4KXAERTTIPHCuRwEXm1L5UGWLCpey6bXU46Iggf3QSLSi5Zl8v14DXiPJeQwF+jXxAfGblEtOJM/Ndo3Vnm9+M0YouNB4p7Jul69qLqyl1yp/BjMpd6I7cX67hX0kr'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0a053-85f4-7113-9241-494da402ed4d-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_7502490', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 'output_tokens': 16, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}
Tool : get_weather
Args: {'location': 'Boston'}


### Tool Execution loop

In [16]:
messages = [{"role":"user", "content":"What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny.


In [17]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'call_1513732': 'El4KXAERTTIP/8QrQJ0WwIyiR+GV1ImfEYx+U7ezaGpy+m4X71GEdMsRLEGphpdQ+Dd0hai/e9TcHfWsLPhx9wd2eYCqyrD5GiBpP/p4IMv9RtHvibAvpu1l2Df1vVGN'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0a0a1-2bd9-7ee1-9b3c-707d10b4c452-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_1513732', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 16, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content="It's sunny in Boston", name='get_weather', tool_call_id='call_1513732')]